In [ ]:
# Colab 請選 2026.07；使用獨立環境以免覆寫已載入的 NumPy。
from pathlib import Path
import subprocess, sys, tempfile, os
assert sys.version_info[:3] == (3, 12, 13), "請使用 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch14-chembl-env")
    if not (env / "bin/python").exists():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["numpy==1.26.4", "pandas==3.0.6", "matplotlib==3.11.2", "rdkit==2026.9.1", "scikit-learn==1.9.1"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "14-01-chembl-case-study.py"
os.environ["MPLCONFIGDIR"] = str(Path(work.name) / "mpl")


# 從 ChEMBL 活性資料到候選分子

Apsel et al. (2008), DOI: [10.1038/nchembio.117](https://doi.org/10.1038/nchembio.117)。資料為 ChEMBL 37 的單一 assay CHEMBL1243977；209 筆全部投影，2026-11-06 取得。重組 EGFR、放射性磷酸轉移測試、10 µM ATP。ChEMBL confidence=8（homologous assignment）、assay organism 未填；不把它描述為已確認的人類野生型 assay，也不混入細胞或其他突變資料。

[全量活動 API](https://www.ebi.ac.uk/chembl/api/data/activity.json?assay_chembl_id=CHEMBL1243977&limit=1000) · [assay 中繼資料](https://www.ebi.ac.uk/chembl/api/data/assay/CHEMBL1243977.json)。資料及投影採 [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/)，署名 ChEMBL / EMBL-EBI 與原研究作者；程式為本書新寫。完整來源欄位投影以 gzip/base64 內嵌，先驗 SHA-256，安裝以外不需連網。

保留 '='、正值、nM 與沒有品質註記的 IC50，另依原文濃度範圍 0.001–50 µM 保守排除範圍外數值，不猜測它們是否設限。167 筆等號值中有 30 筆高於上限、0 筆低於下限；最後137結構、50骨架。同標準化結構以中位 pIC50 聚合，本例沒有合併紀錄。移除39筆缺數值、3筆設限值；這會改變樣本分布，不代表那些資料沒有價值。

固定 GroupShuffleSplit seed42/43，模型只看訓練標籤，選擇只看驗證MAE。先輸出不含觀測值的排名，再揭露測試標籤。這是回溯演練，不是盲態試驗，也沒有發現新活性。第4格同時提供圖與 CSV 下載；完整原值可在教材的 data/ch14/chembl/snapshot.json 查閱。


In [ ]:
%%writefile $script
import base64, gzip, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
SNAPSHOT = 'H4sIAAAAAAACE9WdeW/byBXAv4qg/UdCnezcR1AXjdl1s+1GKvZAgaaBQVO0zVYmXR3Beot8986Qku9kPUz3vcdFNoFISjx+fDNv3vnfcV5sqg/VpirX41fv/rv/eH1SNJeXZb0Zv6q3y+XB7fZqMX4llWJWmLB1vc7DoRfl5emy3TPO3nzz9ug7LpT01o73R3zIV1Veb07yoijX66qpb3723u7L7Sbf3Nlb5HVTV0W+PFlfVst4heMsm2TTmtfF5EUhimIyn4a/jlbTQoQ/xWQ2rYu6EDyceZFv8vDLy2rxxO0smmIbPz997UwqFn7hslmWxXZZPn2QEE6Hg66aTfidKlzjYnu1DBe7Kcev2MF4vcnrRb5anKzK5e6mxofjO9s311fhyPG3mWZ3N2/rahNvtH57d2u4j208Wij/ko0/HqSAUgwF1DybF7xoOdX1JGLLpoWMeAKlQsgAruBHKxhSBpyUkoIlo9IKBdWtPHWyFKUKXJ648B6cEmfxv2ROxmFwmhW8pRHEKtDid2nxaV2LjGdZBjT0WXBUUqhkTk6Q5QQDSlr4kU9IkwzKU1Amiskx0tgXOMELFFfpI5/jOMrEu+yPb95PshkVWm4gM5UTCosXJWDKwAMTpg8viaNZfBVnJCKoELRA1weVFnioiLCyDJ4V78PKGCKK4DGKvq4Nghroe3ByjCanGRQnOwxOXlGVJyhQ8AqF5C6Vk2cOZ10VSUxq0c5R8g4nGf7s5ygppjCwjIYf/aRJXgR7IQgKVbvyCjhnUKzgR0Bn00dALw1NWMC0EBbC2vTApRlh2ZrD0LIKnpZyPWAZRVm2wGh5FJtFMi3ryOqCQNqg1fDrYGt1MilPUcFolcUZECmn1DAst+FLOC6RAKveGZaKQgRFndOwNTmlh0JOkDRiFLwJu3m3qyhAXPrKiaFAkziOkmfEyBzDkJLwa2Rn0jkpR8JTnC2xwmSUQ5jCtEsGZQQdTzEeLW39ENZcnFlDyFGMxotpBAXRc5XMyyCF3z5jvpplk8P5dB73xgNgxkQOPybadGhIobgPJy+8GE+NEDmjdTooTcICVRfyXf3mfUAmYT1czHHJBYb5KZmUIaEPhjGwpQQtUVI6hCWW4emccEJxu/nqZsKSQXrkwxmLzzMYUBwhFDddEzQkYnHlLrenGwHhJypj4fVAlR6Qyy2nYWyqA7AaeppqOSGYBUUPA5NFisTdm/te1AFVdick46l4jLqz+wZJjGgP5xAEbXLM0x++XCXsYyJ0xlJQNVA0d8msh/dxea2SITmOnULXCheGLhhWwRrBiPtp0+B41mxGfyo35eqyqsvF+CleXmPOX1G5uB0R43DYLnt+80GPs0T1ojv1M1E9OHiHKm7tzckzjzd9tep7uxJ+oLmDoFJiYKiERNQ0PsEKCJUcGCppcZXCMAy2rMSeFdDwp4Y2/GlOUKYyGFZ6YKyMRtPT48oJfcYSihEi9glF3VtPwhcSjUwItS4UvDHQC/Elmrr3ko6mvrNdwMxWYmAaoGAM01JRoFoqAq7UDGJ0XAJHuXgcMXgnXBDKde9JCdfTk5VgUqOpf50bpFXVb0IuYNX18IIqUra/58iUomivmGeA0bcBmx6AaBlJERTIRIVRPkGkOxkFsxaZUT3JpkiMECLLPOvByHN8Rk8FBIJwCmMdfI63MTKVEmdItolH8etYeTycIdT3McmxmoJzT8fbewyPicPPTdKKdExS0ki26pZOx8B1SDjHCHzuIU3KkqkWfYwy7HmEwGdrkjkZTkKcbkhBF/bhCEWIrebJnKxGTP+I+QOf0/lgahCzYRSfE9xpAqHPu7y4aTZHiX9WQjo2iEjNOATQ4IUOjA+iEqcISiCR3AKk5ALFtEZI35Y9UAkkPXD+a4ZZmHqpfBjlEIRQHJ3TJ+1JQKzg86q01+mkNI52cXi/BvE8ay1KGcbaSiOUjNbMpaMynoJQIU9UagiNDoRwkgwrLFjCaARbLTfpsLylWQyhHRChYBn49ZVOr1whJOeI2vpnzRYwoKzmGOWI00EJjTgEfr7GCBQpBB0whu8lo5KeAKrHlgo4UPDLX8d4OieNZqmoPytScaKqgeo4moGsgaWxBMxKTy6BYUBZhKaZn8H0jKg/6TilbNI5VIS61AOI9JNek8j4mOGVsEXoQsbTYyoU82Rry8FUk8NIo083/SkhCdcAjJ4rGFrwCoWWPaQKJ9d39izbHwwohKbByRGaSnN8oXoaE9DwpxH8vu6Lkj2U0ciihZXqK1NbW2HnuimctN8ZfnEKmdrZCh0VoYTfybvZ797HssLv5i/eTwEL9cjUDlfY1DSzhNK072ODo+Y8/fWwRkr7fZRhkOGUgh5KbxChpSYCKvtqhoNKYpRX70FK0Sgq0oFCaV+AUA5V9iGF1B7ksUzhiJRB6AvSc/SzhoRMxWZWWLAwQtf7wfI0Oky0oFAyrbhFCLJ16T3UhWGKDKoahdNQ9D/DSfSYwIAk/VBGPiMJKBQ1EqShpBUYhaVLHM6nJMIqwrAnh5EGZwwRVSKiw1ElHEJIbZ/aL8ZSUCU6DR2TFtIwmIzLEelhGvC0uGK7RQxeQ1EALRPI0xZ2tj33CGGbwvVAxSlYK7phsEFoDiwRUuF8em6BlQyvoOYnRWld8Aym6ZFHCK71PYwUVkkaZtrZD7G6epCqQxRnlUcI3ew5V2ka5tp7wBCCbZnEKDfSZ8ayjIaI/RD4oElY4DUUI4Z1NAy3d3mhABuM+u4dEV9jxDXL5OEsy36QKGXoEDrEaeu/JLjTcUEnDK3rbV+obkWm2zQfHcRPAcakBYgDC/p0wtBBOG/pASMTA+v3J5xilLq93DIDRKYHhkwr7Nank3l220kTYXoTbGgjo3GUxCzDkLNU7yU6MyeoMWst+JFb/BeQ3MD6YjlvaBQL37eI7qobgzaFkW5g0DzH0URaTZ9CX+/ITA6MmVAkBY0DVRFXdmjAJJ6JJKMhZMoq+jlDXnkagtVqHLfdAjlQfX6uMUKxeXptBm8kWVAZUIYDQkcmm8zJWoqcuurvAdYMCpYdRis67zlRXi2uORQuBHe0Tq54LBnTpKULCJc1bghFjyXjNLWLdtaaAaGCD6AKync6KinJogKasyxC+1SRXvRTMmUJSxUMK6cRSt+ltxOUzHDCYgXFaiCpyZJZTZMWpGh5JRGGQd2DlvOEcpORwkjhu18Ik46KMxJhvxO0XCKPUUeDqXRQHLPzRUz4+lw+5TFM528E0wUTPYRKcipZem3EKHxTaSbgtUDje5BSmmqLEhhMfhDmCq7J1qmew8SBCoT0ZJeMyaK353yy5SNIaQY2kPoZkjsK3S6mSJQGknsiBUPSIe4HeWJ1UOAcfq1rlE7HxDXV1jFAMxPCOpf7dE7CE+4cDRMxzRG6/NhkUIpKi+8oVGisBIIRPV2otCXBaobHaRjlcqWwHJ9U54yXGSIvh+Ce72OWFU5TbiANwwreNmFN8ppXeE9WAcxgckPgF76SpTt8JZc0QCGykgi1qI1UX9LpUVpFMpXn6+ww+xoynUenBsEgZxpIi1RW8isiaQbacvJpBtJiFZTshkXRjoyyMwiq/ZAoY8Z+FD0ITAahmKTgPXR4S6OYpOy89TgyZRAqSbpkndAxCi0vdiNfQNNW/YQGJQdiaHcU6kjWiPOU1QoFVTIoySgogvUDRRAszyqAQijplB6p5BR6dZIw6KEJE3wuXLgw9kVLLKcdNrGbMheFxGnY7oe1uHKWUK20FxiFtpwe2HrYOQq10aJNd48rA2vTqdTAYHlGpyrajlYBSIsPjBZX6MEyeAq800PDJXD0jUMawLhybmDAlCBiekIjNjD90CMWh6cBTXJphzYuWkbR1QVXSUY7NTBijmT5M0C/ZGCm6bu5vHdEONU4kmUcjpcr0XioGE4N+OyhN7IL0njokayBWA2k34JigkQLGlnv6+JKlJ7hcii4FCOCq/WiYHSiRnB1uT6ktCLhlEQkBe/rsr1kyjgiMtWmPrZNS3CAwTu+rOrBy9EwZdzSwpm1HEKAhrE9eJGp5H6fGLQC74waQqq+4pyKjhH1dhzZ8soPovO74kKRgYVHSzMUVSMZlqSiady0R5vfyWdAaUw4lJUX14IQOyxaaii0jEGPYGsiKSRlXnBucVAlk3I0tA1kWgglV30fWp6Kt+QOL+goXs4HUnQ1jNhEEk5qpMAaweCHQaH7kBJU4jNqLFJuKEIlDaqD67MeLhgHV6Dlh2GGD8KIqwjWO7MTilQxbRBq4/Ya/wwNdwkeKyE8/Fo4vZuFEtbRMb9jkUJYB1uTTMoLSoZ3jIQ74RGckFKlkpKMRhRGMQn6n8ymKH1T4WXKa51MSmDpE/ejm1QUIfU4vgmIFUIaq/TJrKQiM1PVWN2IDUILznT1TypHaKqqMaYqhRAvoy1PJmWIGCqiHHW1wlEaEBuERj42GZYloldgksKxACaj8mQyR/awoEdAbQVCvchkUoopIr1hMOVKW4kSIZNMi6NXxpBFgYWJaWNQPIvJmCQV/weiRCEU4hfppAhV80SaqAxCJKdLNlUoQyGyopZICQiBEvyCSvXxfSiriCypshhQgTYAGoS5ykiXjItGFc8bq3osPw1eIw1+UaXSi/AHaSTiAbmBxSV4yUE3iKwDzUm5QB7ERmOUBEJIIbbJNnYtGWaVknkLipRzxGkEN75PtuNqpcjE3aLUA0JoeuFNsmdYa0co4BY+aU55juAVSV5raUskO6SpkaQJoaq77KFlOEMrdxhrscU1gmfYJzcTjEo/FbVQTLNsLhFAwa+19JeVo1aGK0K1VyeFDIriXAEVX2XaWPL1tJQRjo5gzZEkyxELjn6OaCEV8nzUlWSN4MzyagCCpQ2N1jGdNReFFDduKJnCxiB2fxSdgaKrwX887WqgtQNh7CeYZbOw0JoDrLS8TE3n+f2X40om5bAKC7aMAqvAJ+rss9ZVMm1HwQAsilc4QARck4AraIkiCOLL37zFqg5ipjhDiENLJueReuFGINPYrbjNFOaRWrfyir0fxT//eYOzE79CBpT8N+/jqaUeRPKc4UiOyRtexWTeZTzO7pKKg2SQxONp+HPcjpdx/VwAZD5Kmbpo/sP/ZS3Wknu/e5TjVwFhtfg5XtZ3f2I3AOJ1nDer6/vPvSiXy92J723+VZ7FMvxbncXbC7cSru/d/gJOzlfN9ur+7zWr87yu1pf3t17lq/wyqrP3vr7erPLqweux3p7GK90u89XJ2Sq+iQ9foE3+c3u597aV680TN7ep1uvtw23dsz8a391wsijXxaq62sE6qupFVZ+HQ07z5uSsWV3mUSk/ej0/iRBk6/WMu5b5abkMe9bh6GU5ulqFN6KqR7tvhJc6PvQ7j3j3qjf1WbUo6+Lhed804aVrzpvtevTgFzf56rzcjCKJ825BcOdH1kWzCvcU1nH3f+7b+qI6reKHUXM2WpVBCk6rOgjf6Js/H38/Or0erfJF1bTiHk510azD/4FJvT4rV6P26YzCua9W5TqeJ/4IZ6Pt29HrH/82fqYU7UVgfVFdffpuP32b976/Y/cmiseqOOn43T7XuC1+4gfj7oeeuDLBolemezEek9kPTuvyP9t4y3u1/+ZeW5nbbi6a+CaPX1+ty+Xo6GB0tMyr1egvrw9Gf27qX/Jl+UvcOst/qc5GP749GB2Xy8VlXo/efnMwel39Utbr9nbDMW+as7O45/uD0d+r5bLKL9ej7787GP1w0fw734z+Gr7817o6v9iM/vH6ZcTeXXJ4LGW+LuPlPNyyu9GTViMoVmX7+E4W7eAU5mzOXzD9Qphxd1f7Z/q3n46++zZ7/eO389lz0S6aqtW2X3Im3dd1e2jVvOwqzoSduy/vn+1ZtQpCepWft+EWrTl/J55jHj/8q9mu6jxK0yzceBa+OzqqmuXtnpOzbRzCqs2y7A7arspRPEecNUbh1OFluo7PKAxYd07UarR5HKnvgL7anl6Wi+5tcU55b/n992d3lh/b96hcjK6a5fXVRR7kumjP82q0qILcfShX11EuFttwCVUnb+HdiJs216smSHE5CuP4XriqOmzaBLEd/TsI4rpcx8v90CzDw46RDeHDdZmvxq8EY+5jvOzz8iSMm3nkvKwuq/D+xdXNwbguf76ZzcIbtC437cwTRPVDFQRqv2vTbMJzK5ptfHMF8x+jQG1WVfmhXLRvgzAv4gthuglnE795oxdUcRIWSgvLoo9w9yosTk/CTXfz+ji7aF81aR+9mbcvXDhFeOEYb8esy6twLXFiDEPWIvy8dEKZGEcdnmZVF5uT/SHx1F5wHvOBAqzl7fTDWUwsur3e8U9xMOoEPu52WouPH/8H92m7CSVlAQA='
raw = gzip.decompress(base64.b64decode(SNAPSHOT))
assert hashlib.sha256(raw).hexdigest() == 'a04b7320b86681ece8c6649723eb63d5a399879eef738d0b23c22f156432cad7'
source = json.loads(raw)
assert source['page_meta']['next'] is None
assert source['assay']['assay_chembl_id'] == 'CHEMBL1243977'
assert source['assay']['variant_sequence'] is None
assert source['assay']['bao_label'] == 'single protein format'
assert source['assay']['confidence_score'] == 8  # Homologous assignment; species unresolved.
out = Path('outputs'); out.mkdir(exist_ok=True)
figdir = Path('figures'); figdir.mkdir(exist_ok=True)
df = pd.DataFrame(source['activities']).sort_values('activity_id').reset_index(drop=True)
assert len(df) == source['page_meta']['total_count'] == 209
assert df.activity_id.is_unique
assert set(df.assay_chembl_id) == {'CHEMBL1243977'}
assert set(df.document_chembl_id) == {'CHEMBL1240340'}
audit = []; retained = []
uncharger = rdMolStandardize.Uncharger()
for row in df.to_dict('records'):
    reason = 'retained'
    value = pd.to_numeric(row['standard_value'], errors='coerce')
    if row['standard_type'] != 'IC50': reason = 'wrong_endpoint'
    elif not np.isfinite(value): reason = 'missing_numeric_value'
    elif row['standard_relation'] != '=': reason = 'censored_relation'
    elif row['standard_units'] != 'nM': reason = 'wrong_unit'
    elif value <= 0: reason = 'nonpositive_value'
    elif value < 1: reason = 'below_reported_assay_range'
    elif value > 50000: reason = 'above_reported_assay_range'
    elif row['data_validity_comment'] not in (None, ''): reason = 'validity_flag'
    elif row['potential_duplicate']: reason = 'potential_duplicate'
    elif row['assay_variant_mutation'] is not None: reason = 'variant_annotation'
    else:
        mol = Chem.MolFromSmiles(row['canonical_smiles'] or '')
        if mol is None or mol.GetNumAtoms() == 0: reason = 'invalid_structure'
        else:
            mol = uncharger.uncharge(rdMolStandardize.FragmentParent(mol))
            Chem.SanitizeMol(mol)
            row['smiles'] = Chem.MolToSmiles(mol, isomericSmiles=True)
            row['scaffold'] = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
            row['pIC50'] = 9 - np.log10(value)
            row['IC50_nM'] = float(value)
            retained.append(row)
    audit.append({'activity_id':row['activity_id'], 'molecule_chembl_id':row['molecule_chembl_id'], 'reason':reason})
audit = pd.DataFrame(audit); audit.to_csv(out/'filter_audit.csv',index=False)
clean = pd.DataFrame(retained)
# Same standardized structure within this assay: median pIC50, retaining provenance.
clean = clean.groupby('smiles',sort=True).agg(
    molecule_chembl_id=('molecule_chembl_id',lambda x:';'.join(sorted(set(x)))),
    activity_ids=('activity_id',lambda x:';'.join(map(str,sorted(x)))),
    scaffold=('scaffold','first'), pIC50=('pIC50','median'),
    n_records=('activity_id','size'), pIC50_min=('pIC50','min'), pIC50_max=('pIC50','max')
).reset_index()
assert clean.smiles.is_unique
mols = [Chem.MolFromSmiles(s) for s in clean.smiles]
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048,includeChirality=True)
fps = [generator.GetFingerprint(m) for m in mols]
X = np.array(fps,dtype=np.uint8)
groups = clean.scaffold.to_numpy()
# Fractions refer to scaffold groups, not rows. Choose once before examining labels.
develop, test = next(GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42).split(X,groups=groups))
tr,va = next(GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=43).split(X[develop],groups=groups[develop]))
train, valid = develop[tr], develop[va]
for a,b in [(train,valid),(train,test),(valid,test)]:
    assert not (set(groups[a]) & set(groups[b]))
clean['split'] = ''
for name,idx in [('train',train),('valid',valid),('test',test)]:clean.loc[idx,'split']=name
# Only train/valid labels enter fitting and selection. The test labels remain sealed below.
y_train = clean.loc[train,'pIC50'].to_numpy(); y_valid = clean.loc[valid,'pIC50'].to_numpy()
models = {'Median':DummyRegressor(strategy='median'),
          'Ridge':Ridge(alpha=10),
          'Random forest':RandomForestRegressor(n_estimators=300,min_samples_leaf=2,random_state=42,n_jobs=1)}
validation=[]
for name,model in models.items():
    model.fit(X[train],y_train)
    validation.append({'model':name,'valid_MAE':mean_absolute_error(y_valid,model.predict(X[valid]))})
validation=pd.DataFrame(validation)
chosen=validation.sort_values(['valid_MAE','model']).iloc[0]['model']
# Applicability flag fixed exclusively from training structures (leave self out).
loo=[]
for i in train:
    sims=DataStructs.BulkTanimotoSimilarity(fps[i],[fps[j] for j in train if j != i])
    loo.append(max(sims))
threshold=float(np.quantile(loo,0.1))
ranked=clean.loc[test,['molecule_chembl_id','smiles','activity_ids']].copy()
ranked['prediction']=models[chosen].predict(X[test])
ranked['nearest_train_similarity']=[max(DataStructs.BulkTanimotoSimilarity(fps[i],[fps[j] for j in train])) for i in test]
ranked['in_domain']=ranked.nearest_train_similarity >= threshold
ranked=ranked.sort_values(['prediction','molecule_chembl_id'],ascending=[False,True])
ranked['rank']=np.arange(1,len(ranked)+1)
# Freeze the entire ranking and top 5 before exposing any test labels.
ranked.to_csv(out/'ranking_before_reveal.csv',index=False)
selected_ids=ranked.head(5).index.to_list()
print('Selected using validation only:', chosen)
print('Frozen top 5, before label reveal:\n', ranked.head(5).drop(columns=['smiles','activity_ids']).to_string(index=False))
# Final evaluation: no model fitting, threshold changes or reranking after this line.
y_test=clean.loc[test,'pIC50'].to_numpy()
metrics=[]
for name,model in models.items():
    pred=model.predict(X[test])
    metrics.append({'model':name,'test_MAE':mean_absolute_error(y_test,pred),'test_RMSE':root_mean_squared_error(y_test,pred)})
metrics=validation.merge(pd.DataFrame(metrics),on='model');metrics.to_csv(out/'metrics.csv',index=False)
ranked['observed_pIC50']=clean.loc[ranked.index,'pIC50']
ranked['observed_IC50_nM']=10**(9-ranked.observed_pIC50)
ranked['absolute_error']=abs(ranked.prediction-ranked.observed_pIC50)
ranked.to_csv(out/'ranking_after_reveal.csv',index=False)
clean.to_csv(out/'curated.csv',index=False)
print('\nFilters:\n', audit.reason.value_counts().to_string())
print('\nSplits:\n', clean.groupby('split').agg(n=('smiles','size'),scaffolds=('scaffold','nunique')).to_string())
print('\nMetrics:\n',metrics.to_string(index=False))
print('\nRevealed top 5:\n',ranked.head(5).drop(columns=['smiles','activity_ids']).to_string(index=False))
print('Domain threshold:',threshold,'test flags:',ranked.in_domain.value_counts().to_dict())
print('Unique structures:',len(clean),'scaffolds:',clean.scaffold.nunique(),'combined records:',int((clean.n_records-1).sum()))
print('Train target range:',float(y_train.min()),float(y_train.max()))
summary={'chosen_model':chosen,'threshold':threshold,'rows':len(clean),'scaffolds':clean.scaffold.nunique(),
         'splits':{k:int((clean.split==k).sum()) for k in ['train','valid','test']},
         'top5_mean':float(ranked.head(5).observed_pIC50.mean()),'test_mean':float(y_test.mean()),
         'top5_ge7':int((ranked.head(5).observed_pIC50>=7).sum()),'test_ge7':int((y_test>=7).sum()),
         'training_range':[float(y_train.min()),float(y_train.max())]}
(out/'summary.json').write_text(json.dumps(summary,indent=2))
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(1,3,figsize=(13,3.8),layout='constrained')
colors={'train':'#287d8e','valid':'#edae49','test':'#c55353'}
for name,idx in [('train',train),('valid',valid),('test',test)]:
    ax[0].hist(clean.loc[idx,'pIC50'],bins=np.linspace(3.5,9.5,13),alpha=0.5,label=f'{name} (n={len(idx)})',color=colors[name])
ax[0].set(xlabel='Observed pIC50',ylabel='Molecules',title='A. Scaffold-held-out data');ax[0].legend(fontsize=8)
x=np.arange(len(metrics));ax[1].bar(x-.18,metrics.valid_MAE,.36,label='Validation',color='#edae49');ax[1].bar(x+.18,metrics.test_MAE,.36,label='Test',color='#c55353')
ax[1].set_xticks(x,metrics.model);ax[1].set(ylabel='MAE (pIC50)',title='B. Select on validation only');ax[1].legend(fontsize=8)
for flag,marker,color in [(True,'o','#287d8e'),(False,'x','#c55353')]:
    z=ranked[ranked.in_domain==flag];ax[2].scatter(z.observed_pIC50,z.prediction,marker=marker,c=color,label='Above cutoff' if flag else 'Below cutoff',alpha=.8)
lo=min(y_test.min(),ranked.prediction.min())-.2;hi=max(y_test.max(),ranked.prediction.max())+.2
ax[2].plot([lo,hi],[lo,hi],'--',color='gray',lw=1)
ax[2].set(xlabel='Observed pIC50',ylabel='Predicted pIC50',title=f'C. {chosen}: final test');ax[2].legend(fontsize=8)
fig.savefig(figdir/'14-01-chembl-evaluation.png',dpi=160);plt.close(fig)
fig,ax=plt.subplots(1,2,figsize=(10,4),layout='constrained')
top=ranked.head(5);x=np.arange(5)
ax[0].plot(x,top.prediction,'o-',label='Predicted',color='#287d8e');ax[0].plot(x,top.observed_pIC50,'s--',label='Observed after selection',color='#c55353')
ax[0].set_xticks(x,[str(i) for i in top['rank']]);ax[0].set(xlabel='Frozen candidate rank',ylabel='pIC50',title='A. Five selected candidates');ax[0].legend(fontsize=8)
ax[1].scatter(ranked.nearest_train_similarity,ranked.absolute_error,color='#287d8e',alpha=.75);ax[1].axvline(threshold,color='#c55353',ls='--',label=f'Train-only cutoff = {threshold:.3f}')
ax[1].set(xlabel='Nearest training Tanimoto similarity',ylabel='Absolute error (pIC50)',title='B. Similarity is not an error bound');ax[1].legend(fontsize=8)
fig.savefig(figdir/'14-01-chembl-ranking.png',dpi=160);plt.close(fig)


In [ ]:
subprocess.run([python, str(script)], cwd=work.name, check=True)
from IPython.display import display, Image, FileLink
import shutil
for folder in ["figures", "outputs"]:
    dest = Path(folder); dest.mkdir(exist_ok=True)
    for src in sorted((Path(work.name) / folder).iterdir()):
        target = dest / src.name
        shutil.copy2(src, target)
        if src.suffix == ".png":
            display(Image(filename=str(target)))
        else:
            display(FileLink(str(target)))
